In [0]:
#Create autoload_input folder in volume
dbutils.fs.mkdirs("/Volumes/financepnd/bronze/landing/autoloader_input/2010/12/01")
dbutils.fs.mkdirs("/Volumes/financepnd/bronze/landing/autoloader_input/2010/12/02")
dbutils.fs.mkdirs("/Volumes/financepnd/bronze/landing/autoloader_input/2010/12/03")
dbutils.fs.mkdirs("/Volumes/financepnd/bronze/landing/autoloader_input/2010/12/04")
dbutils.fs.mkdirs("/Volumes/financepnd/bronze/landing/autoloader_input/2010/12/05")
dbutils.fs.mkdirs("/Volumes/financepnd/bronze/landing/autoloader_input/2010/12/06")
dbutils.fs.mkdirs("/Volumes/financepnd/bronze/landing/autoloader_input/2010/12/07")

In [0]:
#Create checkpoint location in volume
dbutils.fs.mkdirs("/Volumes/financepnd/bronze/landing/checkpoint/autoloader")

In [0]:
#copy files to nested location
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-01.csv","/Volumes/financepnd/bronze/landing/autoloader_input/2010/12/01")
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-02.csv","/Volumes/financepnd/bronze/landing/autoloader_input/2010/12/02")
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-03.csv","/Volumes/financepnd/bronze/landing/autoloader_input/2010/12/03")


In [0]:
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-03.csv","/Volumes/financepnd/bronze/landing/autoloader_input/2010/12/05")

In [0]:
## Read files using Autoloader with checkpoint
# and schema location "/Volumes/dev/bronze/landing/checkpoint/autoloader"
#File Detection Modes:
# - Directory listing (uses API calls to detect new files)
# - File notification (uses Notification and Queue services - requires elevated cloud permissions for setup)
df = (spark
		.readStream
		.format("cloudFiles")
		.option("cloudFiles.format","csv")
		.option("pathGlobFilter","*.csv")
		.option("header","true")
        .option("cloudFiles.schemaHints","Quantity int,UnitPrice double")
		.option("cloudFiles.schemaLocation","/Volumes/financepnd/bronze/landing/autoloader_input/checkpoint/autoloader/1/")
		.load("/Volumes/financepnd/bronze/landing/autoloader_input/*/")
		)

In [0]:
# Write data to delta table - dev.bronze.invoice_al_1
from pyspark.sql.functions import col
(
df.withColumn("_file",col("_metadata.file_name"))
  .writeStream
  .option("checkpointLocation","/Volumes/financepnd/bronze/landing/autoloader_input/checkpoint/autoloader/1/")
  .outputMode("append")
  .trigger(availableNow=True) #It will process available file and stop, it will run batch mode
  .toTable("financepnd.bronze.invoice_al_1")
)

In [0]:
%sql
select * from financepnd.bronze.invoice_al_1;

In [0]:
%sql
select _file,count(1) from financepnd.bronze.invoice_al_1 group by _file;